# SBSI tutorial: predicting shear response

This notebook walks through the prediction half of SBSI. You load a trained measurement flow and a blending emulator, hand them a catalogue of galaxies, and get back the shear response of that catalogue — the number you calibrate with.

## 1. Setup

Everything you need is exported from the top-level `sbs_shear` package. Run this from the repository root with `PYTHONPATH` set, or from anywhere if you installed SBSI with `pip install -e .`.

## 2. Point SBSI at a catalogue

`load_catalogue` reads a Feather, Parquet, CSV, or pickle path.

The catalogue bundled with this notebook is a truth-level input catalogue. It holds about 111,000 galaxies with sky positions (`RA`, `DEC`), a magnitude (`r`), a size (`Re`), a shape (`sersic_n`, `axis_ratio`, `position_angle`), a `redshift`, and the applied shear (`g1`, `g2`).

If you need fresh simulated or measured catalogues, `job_blendemu.sh` in this folder shows how to call BlendEMU's pipeline. SBSI does not render or measure images itself.

In [ ]:
from pathlib import Path

from sbs_shear import (
    EmulatorPairingConfig,
    ModelPaths,
    ResponsePredictor,
    get_model,
    load_catalogue,
    load_emulator,
    prepare_forward_catalogue,
    predict_blend_response,
)

In [ ]:
EXAMPLES = Path("examples") if Path("examples").is_dir() else Path(".")
INPUT_CATALOGUE = EXAMPLES / "data/example_catalog.feather"
input_catalogue = load_catalogue(INPUT_CATALOGUE)

input_catalogue.head()

## 3. Load the models

In [ ]:
# Frozen release paths:
models = get_model("V3")

# An arbitrary model uses the identical workflow:
custom_models = ModelPaths(
    flow_checkpoints=(
        Path("/path/to/models/flow_s1.pt"),
        Path("/path/to/models/flow_s2.pt"),
    ),
    emulator_model=Path("/path/to/models/emulator.json"),
    emulator_metadata=Path("/path/to/models/emulator_metadata.json"),
)

OBSERVING_CONDITIONS = {
    "pixel_size": 0.2,
    "zero_point": 30.0,
    "psf_fwhm": 0.73,
    "moffat_beta": 2.224,
    "pixel_rms": 0.312,
}

models

## 4. Predict the response

The response has two parts, and you always need both:

$$R_{\mathrm{model}} = R_{\mathrm{flow}} + R_{\mathrm{blend}}.$$

`R_flow` is *self*-response: how an isolated galaxy's measured shape reacts to shear. `R_blend` is the extra response caused by neighbours sitting in the way.

By default `predict` refuses to run if any of your galaxies fall outside the domain, and tells you how many. That is deliberate: extrapolating a flow is not a small error. If you would rather drop those rows and carry on, pass `strict_domain=False`.

In [ ]:
# Load the two models. device=None picks CUDA when available and CPU otherwise;
# use a GPU node for production-scale flow prediction.
emulator = load_emulator(models, conditions=OBSERVING_CONDITIONS, device="cpu")
predictor = ResponsePredictor.load(models, device=None)
print("Condition features:", predictor.condition_features)
print("Measured targets:  ", predictor.target_features)
print("Training domain:   ", predictor.domain)

`prepare_forward_catalogue` runs the neighbour search and hands back both views as ordinary DataFrames you can inspect:

- `flow_inputs` — one row per galaxy that survived the cuts, indexed by its row number in your original catalogue.
- `emulator_pairs` — one row per accepted pair, so a crowded galaxy shows up several times.

(Working across several fields or tiles? Pass `group_column` so a galaxy is never paired with a "neighbour" from a different field.)

In [ ]:
pairing = EmulatorPairingConfig.from_emulator(emulator)
prepared = prepare_forward_catalogue(input_catalogue, config=pairing)
prepared.emulator_pairs[["primary_row", "secondary_row", "distance"]].head()

# The pair-level predictions are summed into a Series keyed by primary_row.
r_blend = predict_blend_response(emulator, prepared.emulator_pairs)

prediction = predictor.predict(
    prepared.flow_inputs,
    blend_response=r_blend,
    shear=0.02,
    n_samples=64,
)
prediction.summary()

### Distribution of the two response channels

`prediction.flow` and `prediction.blend` are per-object arrays: each galaxy's
self response, and the extra response its neighbours contribute. Their
distributions show how the two channels split object by object, beyond the
catalogue means in `prediction.summary()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
channels = (
    (prediction.flow, r"self response $R_{\mathrm{flow}}$"),
    (prediction.blend, r"blending response $R_{\mathrm{blend}}$"),
)
lo = min(values.min() for values, _ in channels)
hi = max(values.max() for values, _ in channels)
bins = np.linspace(lo, hi, 51)
for ax, (values, label) in zip(axes, channels):
    ax.hist(values, bins=bins, density=True, color="0.6")
    ax.axvline(values.mean(), color="crimson", ls="--", lw=1.2)
    ax.set_xlabel(f"{label}\nmean = {values.mean():+.4f}")
axes[0].set_ylabel("probability density")
fig.suptitle("Per-object response channels")
fig.tight_layout()
plt.show()

### Reusing emulator output you already have

If you ran the emulator earlier and saved the result. Hand `predict` the saved catalogue and name the column holding the response; it joins on `(case, input_index)`:

```python
EMULATOR_RESPONSE_CATALOGUE = Path("/path/to/emulator_response.feather")
prediction = predictor.predict(
    prepared.flow_inputs,
    blend_catalogue=EMULATOR_RESPONSE_CATALOGUE,
    blend_response="R_blend",
)
```

And if `R_blend` is already a column of `prepared.flow_inputs`, a plain `predictor.predict(prepared.flow_inputs)` picks it up.

One warning, because it is the easiest way to get a quietly wrong answer: alignment has to come from stable object keys or from the way you built the table. Two DataFrames of the same length are not necessarily in the same order. Rows SBSI cannot match are dropped — they are never treated as `R_blend = 0`.

### Reading the result

`prediction` carries per-object `prediction.flow`, `prediction.blend`, and `prediction.total`, with `prediction.total_mean` for the catalogue average. `prediction.summary()` prints the short version: how many objects and seeds went in, the three mean responses, and `R_flow_seed_sem` — the scatter across the ensemble seeds, which is your handle on how much the flow itself is uncertain.

### Comparing against a simulation

Once you have a matched simulation response, the multiplicative bias is

$$m = R_{\mathrm{sim}} / R_{\mathrm{model}} - 1.$$

```python
m = prediction.multiplicative_bias(simulation_response)
print(f"m = {100 * m:+.3f}%")
```

## 5. Catalogue-level shear inference — not ready yet

The long-term goal is to skip the response-and-bias detour entirely and infer shear directly, putting the measurement flow and the neighbour emulator inside a single simulation-based likelihood.